In [1]:
import pandas as pd
import numpy as np

PATH = "../data/processed/netload/f10_netload_nasa_clean.csv"
df = pd.read_csv(PATH, parse_dates=["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

# keep only what hybrid needs
cols = ["Timestamp", "NetLoad_MW", "ALLSKY_SFC_SW_DWN", "T2M", "WS10M"]
df = df[cols].copy()

df.head(), df.tail(), df.shape

(            Timestamp  NetLoad_MW  ALLSKY_SFC_SW_DWN    T2M  WS10M
 0 2025-03-31 15:45:00      -1.356             425.48  31.07   3.42
 1 2025-03-31 16:00:00      -0.353             194.80  30.73   3.49
 2 2025-03-31 16:15:00       0.509             194.80  30.73   3.49
 3 2025-03-31 16:30:00       1.902             194.80  30.73   3.49
 4 2025-03-31 16:45:00       2.148             194.80  30.73   3.49,
                 Timestamp  NetLoad_MW  ALLSKY_SFC_SW_DWN    T2M  WS10M
 14779 2025-09-01 14:30:00      -0.331             390.60  30.52   4.32
 14780 2025-09-01 14:45:00       0.169             390.60  30.52   4.32
 14781 2025-09-01 15:00:00       0.888             246.32  30.28   4.47
 14782 2025-09-01 15:15:00       1.433             246.32  30.28   4.47
 14783 2025-09-01 15:30:00       1.677             246.32  30.28   4.47,
 (14784, 5))

### Apply the exact same split boundaries

In [2]:
train_end = pd.Timestamp("2025-07-15 00:00:00")
test_start = pd.Timestamp("2025-08-01 00:00:00")

df_train = df[df["Timestamp"] < train_end].copy()
df_val   = df[(df["Timestamp"] >= train_end) & (df["Timestamp"] < test_start)].copy()
df_test  = df[df["Timestamp"] >= test_start].copy()

print("Train:", df_train.shape, df_train["Timestamp"].min(), "→", df_train["Timestamp"].max())
print("Val:  ", df_val.shape,   df_val["Timestamp"].min(),   "→", df_val["Timestamp"].max())
print("Test: ", df_test.shape,  df_test["Timestamp"].min(),  "→", df_test["Timestamp"].max())

Train: (10113, 5) 2025-03-31 15:45:00 → 2025-07-14 23:45:00
Val:   (1632, 5) 2025-07-15 00:00:00 → 2025-07-31 23:45:00
Test:  (3039, 5) 2025-08-01 00:00:00 → 2025-09-01 15:30:00


### Build issue times (one forecast per day at 23:45) + ensure next-day exists

In [3]:
ISSUE_TIME = "23:45"
H = 96  # next day 96 points

def daily_issue_times(df_block, issue_time="23:45"):
    days = pd.to_datetime(df_block["Timestamp"].dt.date.unique())
    issue_ts = [pd.Timestamp(f"{d.date()} {issue_time}") for d in days]
    existing = set(df_block["Timestamp"])
    return [t for t in issue_ts if t in existing]

ts_set = set(df["Timestamp"])

def has_next_day(t, horizon=96):
    start = t + pd.Timedelta(minutes=15)
    times = [start + pd.Timedelta(minutes=15*i) for i in range(horizon)]
    return all(x in ts_set for x in times)

train_issue = [t for t in daily_issue_times(df_train, ISSUE_TIME) if has_next_day(t, H)]
val_issue   = [t for t in daily_issue_times(df_val, ISSUE_TIME)   if has_next_day(t, H)]
test_issue  = [t for t in daily_issue_times(df_test, ISSUE_TIME)  if has_next_day(t, H)]

print("Train issue days:", len(train_issue), "|", train_issue[0], "→", train_issue[-1])
print("Val issue days:  ", len(val_issue),   "|", val_issue[0],   "→", val_issue[-1])
print("Test issue days: ", len(test_issue),  "|", test_issue[0],  "→", test_issue[-1])

Train issue days: 106 | 2025-03-31 23:45:00 → 2025-07-14 23:45:00
Val issue days:   17 | 2025-07-15 23:45:00 → 2025-07-31 23:45:00
Test issue days:  30 | 2025-08-01 23:45:00 → 2025-08-30 23:45:00


In [4]:
t = test_issue[0]
start = t + pd.Timedelta(minutes=15)
end = start + pd.Timedelta(minutes=15*(H-1))
print("Issue:", t)
print("Target range:", start, "→", end)

Issue: 2025-08-01 23:45:00
Target range: 2025-08-02 00:00:00 → 2025-08-02 23:45:00


### ICEEMDAN energy check (run once)

In [5]:
!pip install EMD-signal

Defaulting to user installation because normal site-packages is not writeable
  Using cached emd_signal-1.9.0-py3-none-any.whl.metadata (9.7 kB)
  Using cached numpy-2.4.2-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
Using cached emd_signal-1.9.0-py3-none-any.whl (76 kB)
Using cached numpy-2.4.2-cp312-cp312-win_amd64.whl (12.3 MB)
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
    --------------------------------------- 0.8/36.5 MB 8.5 MB/s eta 0:00:05
   -- ------------------------------------- 1.8/36.5 MB 5.9 MB/s eta 0:00:06
   -- ------------------------------------- 2.6/36.5 MB 5.2 MB/s eta 0:00:07
   ---- ----------------------------------- 3.7/36.5 MB 5.0 MB/s eta 0:00:07
   ----- ---------------------------------- 4.7/36.5 MB 5.1 MB/s eta 0:00:07
   ----- ---------------------------------- 5.2/36.5 MB 5.1 MB/s eta 0:00:07
   ----- ---------------------------------- 5.2/36.5 MB 5.1 MB/s eta


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Program Files\Python312\python.exe -m pip install --upgrade pip


In [7]:
from PyEMD import CEEMDAN
print(" CEEMDAN import OK")

 CEEMDAN import OK


### CEEMDAN energy check on ONE 7-day TRAIN window 

In [8]:
from PyEMD import CEEMDAN
import numpy as np
import pandas as pd

# --- pick a train issue time that is NOT too early ---
t_check = pd.to_datetime(train_issue[10])  # safe pick

W_DAYS = 7
N = W_DAYS * 96  # 672 points

# exact 7-day (672 point) history ending at t_check (inclusive)
start = t_check - pd.Timedelta(minutes=15*(N-1))
idx = pd.date_range(start, t_check, freq="15min")

win = df_train.set_index("Timestamp").reindex(idx)
if win["NetLoad_MW"].isna().any():
    raise ValueError("Missing points in 7-day window. Pick a later t_check.")

x = win["NetLoad_MW"].to_numpy(dtype=float)

ce = CEEMDAN()
ce.random_seed = 42
imfs = ce.ceemdan(x)  # (K, 672)

energies = np.sum(imfs**2, axis=1)
pct = energies / (energies.sum() + 1e-12) * 100

for i, p in enumerate(pct, start=1):
    print(f"IMF{i}: {p:.2f}%")
print("IMF1-IMF3 total %:", float(pct[:3].sum()))

IMF1: 0.35%
IMF2: 6.28%
IMF3: 62.70%
IMF4: 16.01%
IMF5: 1.15%
IMF6: 13.51%
IMF1-IMF3 total %: 69.3311626709565


### Denoise function (DROP_IMFS = 1)

In [9]:
DROP_IMFS = 1  # drop only IMF1

def denoise_ceemdan(x, drop_imfs=1, seed=42):
    """
    CEEMDAN denoise on a past-only window x.
    Reconstruct = sum(IMFs after dropped ones) + residual
    """
    x = np.asarray(x, dtype=float)

    ce = CEEMDAN()
    ce.random_seed = seed
    imfs = ce.ceemdan(x)  # (K, N)

    residual = x - imfs.sum(axis=0)
    kept = imfs[drop_imfs:, :].sum(axis=0) if imfs.shape[0] > drop_imfs else np.zeros_like(x)

    return kept + residual

# quick sanity check on your same 7-day window x
den = denoise_ceemdan(x, drop_imfs=DROP_IMFS)
print("denoised len:", len(den), "| orig len:", len(x))

denoised len: 672 | orig len: 672


### Build 1 sample at one issue time

In [10]:
import numpy as np
import pandas as pd

L = 192          # input length (2 days)
H = 96           # horizon (next day)
D = 7 * 96       # denoise window (7 days)

t = pd.to_datetime(train_issue[10])  # issue time (23:45)

# index by time
df_train_i = (df_train.copy()
              .assign(Timestamp=lambda d: pd.to_datetime(d["Timestamp"]))
              .sort_values("Timestamp")
              .set_index("Timestamp"))

# past 7 days (for denoise)
hist_start = t - pd.Timedelta(minutes=15*(D-1))
hist_idx = pd.date_range(hist_start, t, freq="15min")

# next day (target)
y_idx = pd.date_range(t + pd.Timedelta(minutes=15),
                      t + pd.Timedelta(minutes=15*H),
                      freq="15min")[:H]

assert hist_idx.isin(df_train_i.index).all(), "Missing history points"
assert y_idx.isin(df_train_i.index).all(), "Missing target points"

hist = df_train_i.loc[hist_idx, ["NetLoad_MW", "ALLSKY_SFC_SW_DWN", "T2M", "WS10M"]]
future = df_train_i.loc[y_idx, ["NetLoad_MW"]]

# denoise net load (past only)
x_den = denoise_ceemdan(hist["NetLoad_MW"].to_numpy(), drop_imfs=DROP_IMFS)

# X: last 2 days (denoised netload + exo)
X = np.concatenate([
    x_den[-L:].reshape(L, 1),
    hist[["ALLSKY_SFC_SW_DWN", "T2M", "WS10M"]].to_numpy()[-L:, :]
], axis=1)

# y: next day netload
y = future["NetLoad_MW"].to_numpy().reshape(-1)

print("Issue:", t)
print("X:", X.shape, "y:", y.shape)
print("X range:", hist_idx[-L], "→", hist_idx[-1])
print("y range:", y_idx[0], "→", y_idx[-1])

Issue: 2025-04-10 23:45:00
X: (192, 4) y: (96,)
X range: 2025-04-09 00:00:00 → 2025-04-10 23:45:00
y range: 2025-04-11 00:00:00 → 2025-04-11 23:45:00


### 1) Decompose once per split → add NetLoad_Denoised

In [12]:
import numpy as np
import pandas as pd
from PyEMD import CEEMDAN

DROP_IMFS = 1  # drop only IMF1 (noise-like)

def add_denoised_column(df_block, drop_imfs=1, seed=42):
    dfb = df_block.copy()
    dfb["Timestamp"] = pd.to_datetime(dfb["Timestamp"])
    dfb = dfb.sort_values("Timestamp").reset_index(drop=True)

    x = dfb["NetLoad_MW"].to_numpy(float)   # original netload

    ce = CEEMDAN()
    ce.random_seed = seed                   # reproducible
    imfs = ce.ceemdan(x)                    # decompose whole split

    residual = x - imfs.sum(axis=0)         # trend part
    kept = imfs[drop_imfs:, :].sum(axis=0)  # keep IMF2+
    dfb["NetLoad_Denoised"] = kept + residual  # reconstructed clean signal

    return dfb

# denoise each split separately (no leakage)
df_train_den = add_denoised_column(df_train, DROP_IMFS)
df_val_den   = add_denoised_column(df_val,   DROP_IMFS)
df_test_den  = add_denoised_column(df_test,  DROP_IMFS)

print("Added NetLoad_Denoised:", df_train_den.shape, df_val_den.shape, df_test_den.shape)

Added NetLoad_Denoised: (10113, 6) (1632, 6) (3039, 6)


### 2) Build (X,y) windows fast from NetLoad_Denoised

In [13]:
import numpy as np
import pandas as pd

EXO = ["ALLSKY_SFC_SW_DWN", "T2M", "WS10M"]  # weather inputs

def build_samples_fast(df_den, issue_list, L=192, H=96):
    df_i = df_den.copy()
    df_i["Timestamp"] = pd.to_datetime(df_i["Timestamp"])
    df_i = df_i.sort_values("Timestamp").set_index("Timestamp")

    X_list, y_list, used = [], [], []

    for t in issue_list:
        t = pd.to_datetime(t)

        X_idx = pd.date_range(t - pd.Timedelta(minutes=15*(L-1)), t, freq="15min")  # past 2 days
        y_idx = pd.date_range(t + pd.Timedelta(minutes=15),
                              t + pd.Timedelta(minutes=15*H),
                              freq="15min")[:H]                                    # next day

        if not X_idx.isin(df_i.index).all(): 
            continue
        if not y_idx.isin(df_i.index).all():
            continue

        X_net = df_i.loc[X_idx, ["NetLoad_Denoised"]].to_numpy()  # denoised netload
        X_exo = df_i.loc[X_idx, EXO].to_numpy()                   # weather
        X = np.concatenate([X_net, X_exo], axis=1)                # (192,4)

        y = df_i.loc[y_idx, ["NetLoad_MW"]].to_numpy().reshape(-1)  # true next-day netload

        X_list.append(X); y_list.append(y); used.append(t)

    X_arr = np.stack(X_list) if X_list else np.empty((0, L, 4))
    y_arr = np.stack(y_list) if y_list else np.empty((0, H))
    return X_arr, y_arr, used

# build datasets (no predicting yet)
X_train, y_train, ts_train = build_samples_fast(df_train_den, train_issue)
X_val,   y_val,   ts_val   = build_samples_fast(df_val_den,   val_issue)
X_test,  y_test,  ts_test  = build_samples_fast(df_test_den,  test_issue)

print("Train:", X_train.shape, y_train.shape, "samples:", len(ts_train))
print("Val:  ", X_val.shape,   y_val.shape,   "samples:", len(ts_val))
print("Test: ", X_test.shape,  y_test.shape,  "samples:", len(ts_test))

Train: (103, 192, 4) (103, 96) samples: 103
Val:   (15, 192, 4) (15, 96) samples: 15
Test:  (29, 192, 4) (29, 96) samples: 29
